In [1]:
import urllib.request
from datetime import datetime
import os
import pandas as pd
import re

- Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторн

In [2]:
province_id = 1

while province_id < 28:
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"province_{province_id}_{timestamp}.csv"
    filepath = os.path.join("data", filename)
    # перевіряємо, чи вже є файл з таким ім’ям
    filename_base = f"province_{province_id}"
    files = os.listdir("data")

    found_substring = False
    for s in files:
        if filename_base in s:
            found_substring = True
            break  

    if not found_substring:
        urllib.request.urlretrieve(url, filepath)
        print(f"Завантажено: {filename}")
    else:
        print(f"Файл {filename} вже існує. Пропускаємо...")
    province_id += 1
    

Файл province_1_20250930_131057.csv вже існує. Пропускаємо...
Файл province_2_20250930_131057.csv вже існує. Пропускаємо...
Файл province_3_20250930_131057.csv вже існує. Пропускаємо...
Файл province_4_20250930_131057.csv вже існує. Пропускаємо...
Файл province_5_20250930_131057.csv вже існує. Пропускаємо...
Файл province_6_20250930_131057.csv вже існує. Пропускаємо...
Файл province_7_20250930_131057.csv вже існує. Пропускаємо...
Файл province_8_20250930_131057.csv вже існує. Пропускаємо...
Файл province_9_20250930_131057.csv вже існує. Пропускаємо...
Файл province_10_20250930_131057.csv вже існує. Пропускаємо...
Файл province_11_20250930_131057.csv вже існує. Пропускаємо...
Файл province_12_20250930_131057.csv вже існує. Пропускаємо...
Файл province_13_20250930_131057.csv вже існує. Пропускаємо...
Файл province_14_20250930_131057.csv вже існує. Пропускаємо...
Файл province_15_20250930_131057.csv вже існує. Пропускаємо...
Файл province_16_20250930_131057.csv вже існує. Пропускаємо...
Ф

- Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області

In [3]:
def remove_html_tags(text):
    return re.sub(r'<.*?>', '', str(text))

folder_path = "data"

province_names = {
    1: "Cherkasy",
    2: "Chernihiv",
    3: "Chernivtsi",
    4: "Crimea",
    5: "Dnipropetrovs'k",
    6: "Donets'k",
    7: "Ivano-Frankivs'k",
    8: "Kharkiv",
    9: "Kherson",
    10: "Khmel'nyts'kyy",
    11: "Kiev",
    12: "Kiev City",
    13: "Kirovohrad",
    14: "Luhans'k",
    15: "L'viv",
    16: "Mykolayiv",
    17: "Odessa",
    18: "Poltava",
    19: "Rivne",
    20: "Sevastopol'",
    21: "Sumy",
    22: "Ternopil'",
    23: "Transcarpathia",
    24: "Vinnytsya",
    25: "Volyn",
    26: "Zaporizhzhya",
    27: "Zhytomyr"
}

all_files = [f for f in os.listdir(folder_path)]
sorted_files = sorted(all_files,key=lambda f: int(re.search(r'province_(\d+)_', f).group(1)))

df_list = []
for file in sorted_files:
    match = re.search(r'province_(\d+)_', file)
    province_id = int(match.group(1))

    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path, skiprows=1, index_col=False)

    df = df.map(lambda x: remove_html_tags(x) if isinstance(x, str) else x)
    df.columns = [remove_html_tags(col) for col in df.columns]
    df = df.iloc[:-1]

    df['Province_ID'] = province_id 
    df['Province_Name'] = province_names[province_id]
    df_list.append(df)
final_df = pd.concat(df_list, ignore_index=True)
print(final_df.head(50))

    year  week    SMN     SMT    VCI    TCI    VHI  Province_ID Province_Name
0   1982   1.0  0.053  260.31  45.01  39.46  42.23            1      Cherkasy
1   1982   2.0  0.054  262.29  46.83  31.75  39.29            1      Cherkasy
2   1982   3.0  0.055  263.82  48.13  27.24  37.68            1      Cherkasy
3   1982   4.0  0.053  265.33  46.09  23.91  35.00            1      Cherkasy
4   1982   5.0  0.050  265.66  41.46  26.65  34.06            1      Cherkasy
5   1982   6.0  0.048  266.55  36.56  29.46  33.01            1      Cherkasy
6   1982   7.0  0.048  267.84  32.17  31.14  31.65            1      Cherkasy
7   1982   8.0  0.050  269.30  30.30  32.50  31.40            1      Cherkasy
8   1982   9.0  0.052  270.75  28.23  35.22  31.73            1      Cherkasy
9   1982  10.0  0.056  272.73  25.25  37.63  31.44            1      Cherkasy
10  1982  11.0  0.069  275.46  26.81  34.79  30.80            1      Cherkasy
11  1982  12.0  0.085  278.05  27.87  35.91  31.89            1 

- Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувались за українською абеткою (1 область - Вінницька)

In [4]:
province_names_ukr = {
    1: "Вінницька", 
    2: "Волинська", 
    3: "Дніпропетровська", 
    4: "Донецька", 
    5: "Житомирська", 
    6: "Закарпатська", 
    7: "Запорізька",
    8: "Івано-Франківська", 
    9: "Київська", 
    10: "Кіровоградська", 
    11: "Луганська", 
    12: "Львівська", 
    13: "Миколаївська", 
    14: "м. Київ",
    15: "м. Севастополь",
    16: "Одеська", 
    17: "Полтавська", 
    18: "Рівненська", 
    19: "Сумська", 
    20: "Тернопільська", 
    21: "Харківська", 
    22: "Херсонська", 
    23: "Хмельницька",
    24: "Черкаська", 
    25: "Чернівецька", 
    26: "Чернігівська", 
    27: "Автономна Республіка Крим"
}
province_translation = {
    1: 24,
    2: 25,
    3: 5,
    4: 6,
    5: 27,
    6: 23,
    7: 26,
    8: 7,
    9: 11,
    10: 13,
    11: 14,
    12: 15,
    13: 16,
    14: 12,
    15: 20,
    16: 17,
    17: 18,
    18: 19,
    19: 21,
    20: 22,
    21: 8,
    22: 9,
    23: 10,
    24: 1,
    25: 3,
    26: 2,
    27: 4
}

df_list_ukr = []
for i in range(1, 27):
    df_mod = df_list[province_translation[i] - 1].iloc[:, :-1].copy()
    df_mod["Province_ID"] = i
    df_mod["Province_Name"] = province_names_ukr[i]
    df_list_ukr.append(df_mod)
final_df_ukr = pd.concat(df_list_ukr, ignore_index=True)
print(final_df_ukr.head(50))
    

    year  week    SMN     SMT    VCI    TCI    VHI  Province_ID Province_Name
0   1982   1.0  0.068  263.59  63.47  28.34  45.90            1     Вінницька
1   1982   2.0  0.074  265.78  67.62  23.05  45.34            1     Вінницька
2   1982   3.0  0.076  267.19  69.37  20.40  44.88            1     Вінницька
3   1982   4.0  0.075  268.57  65.26  17.93  41.60            1     Вінницька
4   1982   5.0  0.072  269.24  58.58  20.00  39.29            1     Вінницька
5   1982   6.0  0.071  270.12  52.37  22.93  37.65            1     Вінницька
6   1982   7.0  0.069  271.42  46.52  23.54  35.03            1     Вінницька
7   1982   8.0  0.071  272.73  43.90  25.02  34.46            1     Вінницька
8   1982   9.0  0.071  273.55  40.16  29.93  35.04            1     Вінницька
9   1982  10.0  0.071  274.83  34.51  34.16  34.33            1     Вінницька
10  1982  11.0  0.075  276.70  31.58  34.05  32.82            1     Вінницька
11  1982  12.0  0.082  278.76  27.79  38.59  33.19            1 

- Реалізувати процедури для формування вибірок наступного виду:
  - Ряд VHI для області за вказаний рік;
  - Ряд VHI за вказаний діапазон років для вказаних областей;
  - Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани;

In [5]:
df.columns = df.columns.str.strip()
final_df.columns = final_df.columns.str.strip()

def vhi_for_province_year(df, province_id, year):
    return df[(df['Province_ID'] == province_id) & (df['year'] == str(year))]['VHI']

def vhi_for_provinces_years(df, province_ids, year_start, year_end):
    mask = (
        df['Province_ID'].isin(province_ids) &
        (df['year'].astype(int) >= year_start) &
        (df['year'].astype(int) <= year_end)
    )
    return df[mask][['Province_ID', 'year', 'VHI']]

def vhi_stats(df, province_ids, year_start, year_end):
    mask = (
        df['Province_ID'].isin(province_ids) &
        (df['year'].astype(int) >= year_start) &
        (df['year'].astype(int) <= year_end)
    )
    vhi_series = df[mask]['VHI']
    return {
        'min': vhi_series.min(),
        'max': vhi_series.max(),
        'mean': vhi_series.mean(),
        'median': vhi_series.median()
    }

print(vhi_for_province_year(final_df, 1, 2010).head())
print(vhi_for_provinces_years(final_df, [1, 2], 2010, 2012))
print(vhi_stats(final_df, [1, 2], 2010, 2012))

1456    53.03
1457    52.35
1458    53.70
1459    55.11
1460    55.12
Name: VHI, dtype: float64
      Province_ID  year    VHI
1456            1  2010  53.03
1457            1  2010  52.35
1458            1  2010  53.70
1459            1  2010  55.11
1460            1  2010  55.12
...           ...   ...    ...
3843            2  2012  42.32
3844            2  2012  43.03
3845            2  2012  44.82
3846            2  2012  44.38
3847            2  2012  43.87

[312 rows x 3 columns]
{'min': np.float64(21.83), 'max': np.float64(74.96), 'mean': np.float64(44.78894230769231), 'median': np.float64(43.15)}
